<a href="https://colab.research.google.com/github/MartVASS/MaskArchitectureAnomaly_CourseProject/blob/main/eomt/notebooks/Evaluation_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/MartVASS/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eomt

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip uninstall wandb
!pip install wandb

!wandb login

In [ ]:
import os
import re

file_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/training/mask_classification_panoptic.py"

# Reset file to add patch
!git checkout -- {file_path}

with open(file_path, "r") as f:
    code = f.read()

# new code with flatten and filtration of valid indices [0, 18]
patched_eval_step = """    def eval_step(
        self,
        batch,
        batch_idx=None,
        log_prefix=None,
    ):
        import torch
        import torch.nn.functional as F
        from torchmetrics.classification import MulticlassJaccardIndex

        # Init mIoU calculator (19 classes)
        if not hasattr(self, "semantic_miou_metric"):
            self.semantic_miou_metric = MulticlassJaccardIndex(num_classes=19).to(self.device)

        # Init mapping tensor COCO -> Cityscapes
        if not hasattr(self, "coco_to_cityscapes_map"):
            user_mapping = {100: 0, 123: 1, 129: 2, 109: 3, 110: 3, 111: 3, 112: 3, 131: 3, 117: 4, 9: 6, 11: 7, 116: 8, 125: 8, 88: 8, 102: 9, 103: 9, 119: 10, 0: 11, 2: 13, 7: 14, 5: 15, 6: 16, 3: 17, 1: 18}

            # We use 255 for non mapped value
            full_map = torch.ones(134, dtype=torch.long) * 255
            for coco_id, city_id in user_mapping.items():
                full_map[coco_id] = city_id
            self.coco_to_cityscapes_map = full_map.to(self.device)

        imgs, targets = batch

        img_sizes = [img.shape[-2:] for img in imgs]
        transformed_imgs = self.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = self(transformed_imgs)

        is_crowds = [target["is_crowd"] for target in targets]
        targets = self.to_per_pixel_targets_panoptic(targets)

        for i, (mask_logits, class_logits) in enumerate(
            list(zip(mask_logits_per_layer, class_logits_per_layer))
        ):
            mask_logits = F.interpolate(mask_logits, self.img_size, mode="bilinear")
            mask_logits = self.revert_resize_and_pad_logits_instance_panoptic(
                mask_logits, img_sizes
            )
            preds = self.to_per_pixel_preds_panoptic(
                mask_logits,
                class_logits,
                self.stuff_classes,
                self.mask_thresh,
                self.overlap_thresh,
            )

            # --- Semantic mIoU calculation ---
            if i == len(mask_logits_per_layer) - 1:
                for p, t in zip(preds, targets):
                    # 1. Conversion and flattened en 1D
                    mapped_preds = self.coco_to_cityscapes_map[p.long()].view(-1)
                    clean_targets = t.long().view(-1)

                    # 2. Security filter : we keep only targets valid in Citscapes [0, 18]
                    # We get rid off -1, 255, and non-mapped COCO prediction.
                    valid_mask = (clean_targets >= 0) & (clean_targets < 19) & (mapped_preds != 255)

                    # 3. Input valid pixel to torchmetrics
                    if valid_mask.any():
                        self.semantic_miou_metric.update(mapped_preds[valid_mask], clean_targets[valid_mask])

            self.update_metrics_panoptic(preds, targets, is_crowds, i)

        if batch_idx % 20 == 0:
            current_miou = self.semantic_miou_metric.compute()
            print(f" -> Image {batch_idx:03d}/500 | mIoU Sémantique Cityscapes (Zero-Shot) Actuel : {current_miou*100:.2f}%")
"""

pattern = r"    def eval_step\(.*?self\.update_metrics_panoptic\(preds, targets, is_crowds, i\)"
code_modified, count = re.subn(pattern, patched_eval_step, code, flags=re.DOTALL)

if count > 0:
    with open(file_path, "w") as f:
        f.write(code_modified)
    print("File repatched sucessfully !")
else:
    print("Error during repatch...")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys
import os

sys.path.append(os.getcwd())

data_path = "/content/drive/MyDrive/Cityscapes"
path_to_model = "/content/drive/MyDrive/COCO/eomt_coco.bin"

import torch
import gc

# 1. Clean residual memory of GPU before relaunch.
gc.collect()
torch.cuda.empty_cache()

# 2. Environment configuration to optimize VRAM allocation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python3 main.py validate \
  -c configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml \
  --data.class_path datasets.cityscapes_semantic.CityscapesSemantic \
  --data.path "$data_path" \
  --model.ckpt_path "$path_to_model" \
  --trainer.devices 1 \
  --model.network.masked_attn_enabled False \
  --model.img_size [640,640] \
  --data.img_size [640,640] \
  --data.batch_size 1 \
  --data.num_classes 133 \
  --model.stuff_classes "[]"